# AI RAG Knowledge Assistant

A question-answering assistant that uses Retrieval-Augmented Generation (RAG) to answer questions about an organisation, its employees, and its products.

The application retrieves relevant information from a local knowledge base and supplies it as context to an LLM before generating a response. A Gradio chat interface provides an interactive way to query the knowledge base.

## Features

- Loads employee and product information from local text files.
- Retrieves relevant context using keyword matching.
- Augments user questions with retrieved context.
- Generates answers using the OpenAI Chat Completions API.
- Provides an interactive chat interface with Gradio.

## Project structure

```text
.
├── ai-rag-knowledge-assistant-manish-kundu.ipynb
├── knowledge-base/
│   ├── employees/
│   └── products/
└── .env
```

The `knowledge-base` directory must contain the employee and product text files used by the application.

## 1. Dependencies and configuration

In [ ]:
import glob
import os
from pathlib import Path

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise EnvironmentError("OPENAI_API_KEY is not configured.")

MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
client = OpenAI(api_key=api_key)

## 2. Load the knowledge base

In [ ]:
def load_knowledge_base():
    """Load employee and product documents into a searchable dictionary."""
    knowledge = {}
    document_dirs = [
        Path("knowledge-base/employees"),
        Path("knowledge-base/products"),
    ]

    for directory in document_dirs:
        if not directory.exists():
            raise FileNotFoundError(
                f"Knowledge-base directory not found: {directory}"
            )

        for file_path in directory.glob("*"):
            if not file_path.is_file():
                continue

            if directory.name == "employees":
                key = file_path.stem.split()[-1].lower()
            else:
                key = file_path.stem.lower()

            knowledge[key] = file_path.read_text(encoding="utf-8")

    if not knowledge:
        raise ValueError("No documents were found in the knowledge base.")

    return knowledge


knowledge = load_knowledge_base()
print(f"Loaded {len(knowledge)} knowledge-base documents.")

## 3. Retrieve relevant context

In [ ]:
def get_relevant_context(message):
    """Retrieve documents whose keys appear in the user's question."""
    normalized_text = "".join(
        character for character in message
        if character.isalpha() or character.isspace()
    )
    words = normalized_text.lower().split()

    return [
        knowledge[word]
        for word in dict.fromkeys(words)
        if word in knowledge
    ]


def build_additional_context(message):
    """Format retrieved documents for inclusion in the LLM prompt."""
    relevant_documents = get_relevant_context(message)

    if not relevant_documents:
        return "No additional context was found for the user's question."

    context = "\n\n".join(relevant_documents)
    return (
        "The following information from the knowledge base may be relevant "
        "to the user's question:\n\n"
        f"{context}"
    )

## 4. Configure the assistant

In [ ]:
SYSTEM_PROMPT = """You are an expert assistant for Insurellm, an insurance technology company.
Answer questions about the organisation, its employees, and its products.

Use the supplied context when it is relevant. Give brief, accurate answers.
Do not invent information. If the answer is not available in the context,
clearly say that you do not know.
"""

## 5. Generate responses

In [ ]:
def chat(message, history):
    """Generate an answer using conversation history and retrieved context."""
    context = build_additional_context(message)
    system_message = f"{SYSTEM_PROMPT}\n\n{context}"

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
    )

    return response.choices[0].message.content

## 6. Launch the chat interface

In [ ]:
demo = gr.ChatInterface(
    fn=chat,
    type="messages",
    title="Insurellm Knowledge Assistant",
    description="Ask questions about Insurellm employees and products.",
)

demo.launch()

## Implementation notes

This notebook demonstrates a simple, keyword-based RAG approach. It retrieves documents when their keys appear in the user's question and adds those documents to the LLM prompt.

This is a baseline implementation rather than a semantic retrieval system. It does not use embeddings, vector search, chunking, or a vector database. These techniques can be introduced to improve retrieval as the knowledge base grows.